In [26]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [27]:
# 모든 agent가 접근 가능 (# 모든 agent에게 보내는 메시지나 대화 기록을 포함하는 전역 상태같은 것 )
class AgentsState(MessagesState):
    current_agent : str
    transfered_by : str 

llm = init_chat_model("openai:gpt-4o")

In [28]:
def make_agent(prompt, tools): # state로 그래프 생성 
    def agent_node(state : AgentsState): # 메시지를 보내서 ai를 호출
        llm_with_tools = llm.bind_tools(tools)  
        response = llm.invoke(
            f"""
                {prompt}

                Conversation History:
                {state["messages"]}
            """            
        ) # 예를 들어 너는 한국어 에이전트고, 한국어만 사용하고, 유저가 다른 언어를 쓰면 transfer tool을 사용해라 같은게 가능 
        return {
            "messages" : [response]
        }
    agent_builder = StateGraph(AgentsState)
    agent_builder.add_node("agent", agent_node)
    agent_builder.add_node(
        "tools",
        ToolNode(tools=tools)
    )

    agent_builder.add_edge(START, "agent")
    agent_builder.add_conditional_edges("agent", tools_condition) # tools_condition: 호출 되어야하는 tool있는지 확인하는 미리 만들어진 tool / 호출되어야하는 tool이 잇다면 tools node로 가게된다.
    agent_builder.add_edge("tools", "agent")
    agent_builder.add_edge("agent", END)

    return agent_builder.compile()

In [29]:
@tool 
def handoff_tool(transfer_to :str, transfered_by : str ): # transfer_to: 통제권을 받을 agennt 이름 /transferd_by : 누가 통제권을 넘겼는가 
    # agent에게 이 tool을 언제 써야하는지 알려주는 
    """
    Handoff to another agent.

    Use this tool when the customer speaks a language that you don't understand.

    Possible values for `transfer_to`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Possible values for `transfered_by`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Args:
        transfer_to: The agent to transfer the conversation to
        transfered_by: The agent that transferred the conversation
    """
    return Command(
        update = {
            "current_agent" : transfer_to,
            "transfered_by" : transfered_by 
        },
        goto = transfer_to ,# 가고싶은 노드 ,
        graph = Command.PARENT, # tranfer_to 노드로 보내고 싶은데 현재 그래프가 아닌 부모 그래프에서 보내고 싶다는 뜻 
    )

In [ ]:
graph_builder = StateGraph(AgentsState)

graph_builder.add_node(
    "korean_agent",
    make_agent( # 그래프 넘기는거임 
        prompt="You're a Korean customer support agent. You only speak and understand Korean.",
        tools=[handoff_tool],
    ), 
)
graph_builder.add_node(
    "greek_agent",
    make_agent(
        prompt="You're a Greek customer support agent. You only speak and understand Greek.",
        tools=[handoff_tool],
    ), 
)
graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="You're a Spanish customer support agent. You only speak and understand Spanish.",
        tools=[handoff_tool],
    ), 
)
 
graph_builder.add_edge(START, "korean_agent") # 서로 통제권을 넘길 수 있기 떄문에 시작점은 뭘로 하든 상관 X 

graph = graph_builder.compile()


In [ ]:
for event in graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hola! Necesito ayuda con mi cuenta.",
            }
        ]
    },
    stream_mode="updates",
):
    print(event)

{'korean_agent': {'messages': [HumanMessage(content='Hola! Necesito ayuda con mi cuenta.', additional_kwargs={}, response_metadata={}, id='8b7e0c0f-48d0-4aac-b8e5-ece11b162b13'), AIMessage(content='죄송합니다. 한국어로만 지원 가능합니다. 도움이 필요하신 경우, 한국어로 문의해 주세요. 감사합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 78, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cb0ea620e7', 'id': 'chatcmpl-DJXokbjZB5NwObREGRSVqV0vwtONR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019cefda-753e-70f2-b115-38061ebc4b26-0', usage_metadata={'input_tokens': 78, 'output_tokens': 26, 'total_tokens': 104, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details':